# STA feature-shard smoke test

This notebook performs the first integration check between the feature pipeline and the STA models. It creates the deterministic eight-event contract shard when needed, loads it through the shared validator, and checks its structure.

Later, change `SHARD_PATH` to the real eight-event shard shared with Niels and rerun the same checks.

## 1. Locate the project and import the shard tools

In [1]:
import sys
from pathlib import Path

for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate / "src" / "sta_features.py").exists():
        PROJECT_ROOT = candidate
        break
else:
    raise FileNotFoundError("Could not locate the project root.")

sys.path.insert(0, str(PROJECT_ROOT))

try:
    import torch
except ImportError as error:
    raise ImportError(
        "PyTorch is required. Activate rd_lab and install torch==2.5.1 first."
    ) from error

from scripts.create_sta_example_shard import build_example_shard
from src.sta_features import load_sta_feature_shard, save_sta_feature_shard

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch version: {torch.__version__}")

Project root: C:\Users\Emiel\Documents\RnD\R-D_Group2
PyTorch version: 2.5.1+cpu


## 2. Create or select the eight-event shard

The default is the artificial contract shard. Replace this path with the agreed real shard when it becomes available.

In [2]:
SHARD_PATH = PROJECT_ROOT / "results" / "local_sta_contract" / "train" / "shard_00000.pt"

if not SHARD_PATH.exists():
    save_sta_feature_shard(build_example_shard(), SHARD_PATH)
    print(f"Created contract shard: {SHARD_PATH}")
else:
    print(f"Using existing shard: {SHARD_PATH}")

Using existing shard: C:\Users\Emiel\Documents\RnD\R-D_Group2\results\local_sta_contract\train\shard_00000.pt


## 3. Load and inspect the shard

Loading through `load_sta_feature_shard` validates the complete shared contract before returning the data.

In [3]:
shard = load_sta_feature_shard(SHARD_PATH, expected_split="train")

print("Keys:", sorted(shard))
print("Events:", len(shard["narration_ids"]))
print("Feature shape:", shard["features"].shape)
print("Feature dtype:", shard["features"].dtype)
print("Mask shape:", shard["valid_mask"].shape)
print("Mask dtype:", shard["valid_mask"].dtype)
print("Statuses:", shard["status"])
print("Narration IDs:", shard["narration_ids"])

Keys: ['action_start_seconds', 'features', 'narration_ids', 'noun_labels', 'observation_end_seconds', 'schema_version', 'split', 'status', 'valid_mask', 'verb_labels', 'video_ids']
Events: 8
Feature shape: torch.Size([8, 5, 512])
Feature dtype: torch.float16
Mask shape: torch.Size([8, 5])
Mask dtype: torch.bool
Statuses: ['full', 'partial', 'partial', 'full', 'partial', 'full', 'no_context', 'video_error']
Narration IDs: ['contract_0000', 'contract_0001', 'contract_0002', 'contract_0003', 'contract_0004', 'contract_0005', 'contract_0006', 'contract_0007']


## 4. Run the first acceptance checks

In [4]:
features = shard["features"]
valid_mask = shard["valid_mask"]
event_count = len(shard["narration_ids"])

assert event_count == 8
assert features.shape == (8, 5, 512)
assert features.dtype == torch.float16
assert valid_mask.shape == (8, 5)
assert valid_mask.dtype == torch.bool
assert torch.isfinite(features).all()
assert torch.all(features[~valid_mask] == 0)
assert len(set(shard["narration_ids"])) == event_count
assert {"full", "partial", "no_context", "video_error"}.issubset(shard["status"])

training_features = features.float()
assert training_features.dtype == torch.float32

print("All shard checks passed.")
print("Training feature dtype after conversion:", training_features.dtype)

All shard checks passed.
Training feature dtype after conversion: torch.float32


## Expected result

The shard should contain 8 events, features shaped `[8, 5, 512]` in `float16`, and a boolean mask shaped `[8, 5]`. The last cell should report that all checks passed. Once the real shared shard passes these same cells unchanged, the next task is implementing the masked mean-pooling model.